# Deep Momentum V2 — research master

This notebook is the entry point you can follow in VS Code and rerun with **Run All**.
It calls small, tested worker files in `src/dmv2/`; the loading rules are coded there.

**Selected design:** research initialization in **2000**, rolling **10 years training / 1 year testing**.
Nominal first outcome test: **2010**. The earlier 2010 loader check was only a test date, not a dataset boundary.
The source files still contain their longer histories.

Current Run All performs configuration, fold planning, source checks and one bounded
historical-universe audit, then a bounded monthly-return engineering sample.
Daily prices stay intact for later daily backtesting. Full panels, features and models are not implemented.
No market-capitalization input is used. Protected 2024+ financial values remain outside this workflow.


## 1. Environment and project root

Select **Deep Momentum V2 (.venv)** in the VS Code kernel picker. This bootstrap
finds V2 from its root, the `notebooks/` folder or the Finance-Working-Files repository.
It imports the current local worker code explicitly and enables reloads during development.
It does not install packages, log in, download data or open cloud resources.


In [ ]:
from pathlib import Path
import sys
import json
from dataclasses import asdict
from html import escape
from IPython.display import display, HTML, Markdown

# Clear prior run evidence before setup, so failed reruns cannot display or save old results.
settings = catalog = inspection = audit_path = None
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None

cwd = Path.cwd().resolve()
bases = [cwd, *cwd.parents]
candidates = bases + [p / "Investments_2026" / "deep_momentum_v2" for p in bases]
ROOT = next((p for p in candidates
             if (p / "configs/research_workflow.json").is_file()
             and (p / "src/dmv2/workflow.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the V2 folder in VS Code and restart the notebook kernel.")
SRC = (ROOT / "src").resolve()
for name, module in list(sys.modules.items()):
    if name == "dmv2" or name.startswith("dmv2."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(SRC):
            raise RuntimeError("The kernel has V2 code from another folder. Restart the kernel.")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import dmv2
assert Path(dmv2.__file__).resolve().is_relative_to(SRC)
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")
from dmv2.catalog import SourceCatalog
from dmv2.chronology import build_annual_folds
from dmv2.workflow import (load_workflow, run_inspection, save_inspection,
                            run_monthly_sample, save_monthly_sample)

def show_table(rows, columns):
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(r.get(c, '')))}</td>" for c in columns) + "</tr>"
                   for r in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

print("Project:", ROOT)
print("Python:", sys.executable)
print("Worker code:", Path(dmv2.__file__).parent)


## 2. One settings file

Edit `configs/research_workflow.json`, then rerun this notebook from the top.
The study starts in 2000; the engineering audit date is a separate setting.
`engineering_sample = combined` is a loader demonstration, not a frozen primary modelling universe.
Price sampling is off by default; enabling it reads at most the explicitly configured small ID sample.

The monthly stage is enabled for **January–February 2000 and two requested price IDs**.
Its full monthly cohort ledger is retained. A December 1999 anchor is necessary for
January returns; it does not settle the wider feature-warmup history.

In [ ]:
settings = None
settings = load_workflow(ROOT)
print(json.dumps(settings, indent=2))


## 3. Rolling chronology — calendar plan, not a trained model

These windows refer to **completed outcome years**. A December 2009 signal
predicting January 2010 belongs to the 2010 test; its future label cannot enter
training ending in December 2009. Actual training rows must also have observable labels at the fit cutoff.

The nominal 2000–2009 → 2010 fold needs verified coverage and feature warmup.
Early 2000 momentum inputs may require pre-2000 prices; that convention is still open.
Selecting 2000 does not truncate the raw files or establish 120 complete usable training months.
Rolling classifier history is our chosen adaptation to the paper's expanding history.
The paper's trailing ten-year class-payoff estimator is a separate component.


In [ ]:
training = settings["training"]
folds = build_annual_folds(settings["research_start"], training["last_test_year"],
                           training["train_years"], mode=training["mode"])
show_table([asdict(f) for f in folds],
           ["fold_id", "train_start", "train_end", "label_cutoff", "test_start", "test_end"])
print(len(folds), "nominal annual folds; no features, fits or backtests executed.")


## 4. Local source inventory

The catalog uses manifest-listed files and canonical manifest pins, not directory discovery.
This checks file presence and sizes. It does not repeat the prior whole-file SHA-256
verification or claim complete historical returns. A fresh clone needs your own approved
`configs/data_sources.local.json` and downloaded source releases.


In [ ]:
catalog = None
catalog = SourceCatalog(ROOT)
inventory = catalog.verify(hashes=False)
show_table([asdict(r) for r in inventory],
           ["source_key", "release_id", "object_count", "size_bytes", "hashes_checked"])
print(sum(r.object_count for r in inventory), "pinned local objects present.")


## 5. Bounded historical constituent audit

The default query is one source session, **31 January 2000**, for the S&P 500/400 union.
It does not scan all price history or build a 2000–2023 panel. Source storage is arranged
by security/index history, so many metadata rows may be considered for this one-date result;
the configured row/file/byte limits apply and failures never return truncated success.

`historical_reconstruction` respects supplied date rules. The sources do not certify true
historical knowledge-time identity/revision evidence. Current delisting flags and future
quote availability never select the historical cohort.


In [ ]:
inspection = audit_path = None
inspection = run_inspection(catalog, settings)
if inspection is None:
    display(Markdown("Universe inspection is disabled in the settings file."))
else:
    print("Research initialization:", settings["research_start"])
    print("Engineering snapshot:", inspection.snapshot.as_of)
    print(json.dumps(inspection.snapshot.summary, indent=2))
    show_table(inspection.snapshot.table.select(
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"]).to_pylist()[:12],
        ["source_security_id", "membership_tickers", "indices", "bridge_method",
         "bridge_confidence", "identity_status"])
    print("Preview only; all", inspection.snapshot.table.num_rows, "constituents stay in the ledger.")


### Read the identity statuses correctly

`needs_review` preserves an upstream caution flag. It does **not** mean the stock is
absent or its price history is necessarily wrong. The prior 2010 sample's 755 flagged
links all had provider IDs and successful resolution; 751 used ticker/company matches.
Some warnings can represent harmless aliases; date-specific conflicts need separate evidence.

The notebook retains all original constituents. Do not train on only the unflagged subset
as though it were the full intended universe. Historical identity validity and terminal-event
accounting remain the next data-quality tasks.


## 6. Optional exact-date price check

Set `inspection.run_price_sample` to `true` for the configured small engineering sample.
This cell summarizes the result. Unqueried members remain `not_requested`; unresolved
identity links remain visible. This does not certify whole-history coverage or returns.


In [ ]:
if inspection is None or inspection.prices is None:
    display(Markdown("This optional one-date quote check is disabled. The separate monthly stage below has its own controls."))
else:
    print(json.dumps(inspection.prices.stats, indent=2))
    print(json.dumps(inspection.quote_coverage.summary, indent=2))


## 7. Private audit artifact and reruns

The complete constituent/exclusion ledger and source/request provenance are saved under
ignored `outputs/notebook_audits/`. The same settings/source/code/runtime pins reuse an identical report.
Worker edits produce a separate fingerprinted artifact; changed evidence under otherwise identical pins raises an error.
The versioned notebook is kept without execution outputs; clear outputs before committing it.


In [ ]:
audit_path = None
if inspection is not None and settings["inspection"]["persist_audit"]:
    audit_path = save_inspection(ROOT, inspection)
    print("Private audit:", audit_path)
else:
    print("Private audit persistence is disabled.")


## 8. Monthly returns with the original historical cohort

Each outcome month uses membership at the **preceding source month end**.
January 2000 uses the December 1999 signal cohort; February uses the January cohort.
A removal during the outcome month remains in that month's ledger. Entrants during
the month enter the next signal cohort, without retroactive admission.

The worker computes `provider_adjusted_return = adj_close[end] / adj_close[signal] - 1`
only at exact consecutive independent XNYS month ends, from completed daily bars and
accepted endpoint quality. No last-observed-price fallback, forward fill, dividend
double counting or zero-return imputation is used. Interior daily gaps are audited
separately from endpoint arithmetic. Unrequested prices are distinct from missing prices.

Source adjustments include splits and dividends, but historical identity continuity,
revision availability, merger stitching and terminal cash flows are not certified.
`accounting_certified`, `knowledge_time_verified` and `daily_backtest_ready` remain false.
These are research return observations, not executable portfolio returns or class labels.

The vendor calendar incorrectly lists some NYSE holidays as sessions. The monthly stage
uses pinned `exchange_calendars` XNYS dates and records reconciliation differences.
It keeps the original vendor calendar; missing quotes never define market holidays.

In [ ]:
monthly_run = daily_df = monthly_df = numerical_preview = monthly_artifact_path = calendar_audit = None
monthly_run = run_monthly_sample(catalog, settings)
if monthly_run is None:
    display(Markdown("Monthly return construction is disabled in the settings file."))
else:
    # Preserve the loaded daily OHLCV/action/quality table. Build a separate monthly view.
    daily_df = monthly_run.sample.daily_prices.table.to_pandas().copy(deep=True)
    monthly_df = monthly_run.sample.panel.table.to_pandas().copy(deep=True)
    print("Daily source observations retained:", len(daily_df))
    print("Original-cohort monthly ledger rows:", len(monthly_df))
    show_table([asdict(w) for w in monthly_run.sample.panel.windows],
               ["outcome_month", "signal_date", "outcome_start", "outcome_end"])
    print(json.dumps(monthly_run.sample.panel.summary, indent=2))
    print("Measured cumulative source reads:")
    print(json.dumps(monthly_run.sample.loading_stats["total"], indent=2))
    print("Calendar reconciliation:")
    calendar_audit = monthly_run.sample.loading_provenance["calendar_reconciliation"]
    print(json.dumps({key: calendar_audit[key] for key in
                      ("policy", "package_version", "scope", "source_only_dates",
                       "independent_only_dates", "independent_session_count")}, indent=2))

### Read `monthly_df` before using its numbers

There is one row for **every source member at each signal cutoff**. `return_status`
separates valid ratios, unrequested observations, unresolved identities, missing
endpoints and rejected prices. A missing end can require terminal-event review; it
is not proof of a delisting and never deletes the constituent. The two-ID sample
does not establish complete price coverage of the intended research universe.

`daily_quote_path_complete` audits the independent XNYS daily sessions, independently of
`daily_backtest_ready`. Numeric month-end arithmetic can exist with an interior
gap; future daily backtests must resolve that path and cash-flow accounting.
Monthly volume sums are present only for complete valid daily observations, and
do not represent share turnover.

The `outcome_month` return becomes observable after its month closes. It must not
be a feature at the earlier `signal_date`. No next-month target ranks are built here.
Future feature construction must also retrieve an entrant's legitimate pre-entry
price history; membership selects prediction cohorts, not a truncation of price history.

In [ ]:
numerical_preview = None
if monthly_df is not None:
    numerical_preview = monthly_df.loc[monthly_df["return_status"] == "ok"].head(10)
    show_table(numerical_preview.to_dict("records"),
               ["source_security_id", "signal_date", "outcome_month", "outcome_end",
                "provider_adjusted_return", "return_status", "daily_quote_path_complete"])
    print("Engineering numerical preview only. Full monthly_df retains all", len(monthly_df), "cohort rows.")
    print("daily_df remains the separate daily-frequency source view for later backtesting.")

## 9. Keep daily and monthly tables independently

`daily_df` contains the original bounded daily quotes; `monthly_df` is a separate
derived cohort/return table. Monthly aggregation never overwrites the daily source.
The Arrow source table also remains unchanged. This sample is not the future full
daily execution dataset, and no backtest is run.

Persisted private evidence contains **`daily.parquet`**, **`monthly.parquet`** and
**`audit.json`** with source, code, request and table hashes, plus both original and
independent calendar tables. Identical reruns reuse
it; changed evidence under unchanged provenance is rejected. Clear notebook outputs
before committing; source observations and generated tables remain outside Git.

In [ ]:
monthly_artifact_path = None
if monthly_run is not None and settings["monthly_returns"]["persist"]:
    monthly_artifact_path = save_monthly_sample(ROOT, monthly_run)
    print("Private daily/monthly evidence:", monthly_artifact_path)
else:
    print("Monthly evidence persistence is disabled.")

## 10. What we build next

1. Classify/resolve historical identity warnings with dated evidence.
2. Resolve monthly economic accounting, terminal cash flows, mergers and historically valid identity continuity.
   The provider-adjusted endpoint ratio below is an engineering return, not complete economic certification.
3. Define the feature dictionary, including the 15 paper-derived momentum/state inputs and proposed additions.
4. Construct completed-month labels and class-payoff estimates under explicit cutoff rules.
5. Fit the chosen estimators and evaluate the rolling 10/1 schedule under the frozen experiment plan.

These stages are not implemented yet and are not silently executed by Run All.
The master remains the readable workflow; each implemented stage will call its tested worker module.
See `docs/NOTEBOOK_WORKFLOW.md`, `docs/DATA_FOUNDATIONS.md` and the locked paper reference.
